# 04 - ISTAT processing

Verifica degli indicatori demografici POSAS 2019 e 2025, delle fasce d'età e della comparabilità temporale.

### Riproducibilità

Questo notebook documenta e verifica la fase di elaborazione dei dati demografici ISTAT.

Gli archivi POSAS originali sono conservati in `data/raw/istat/` e le trasformazioni sono implementate negli script della directory `scripts/`.

Il notebook permette di controllare aggregazioni, indicatori demografici, fasce d'età e criteri di comparabilità tra 2019 e 2025.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

def project_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent]
    for c in candidates:
        if (c / "data" / "processed").exists() and (c / "metadata").exists():
            return c
    raise FileNotFoundError("Eseguire il notebook dalla root del repository o da notebooks/.")

ROOT = project_root()
ROOT

## Elaborazione ISTAT POSAS 2019/2025
Il processed `municipality_population.csv` contiene gli indicatori demografici costruiti sulla geografia comunale 2025.

In [2]:
pop = pd.read_csv(ROOT / "data/processed/municipality_population.csv", dtype=str, keep_default_na=False)
assert len(pop) == 7896
assert pop["istat_code"].nunique() == 7896
print(f"Comuni 2025: {len(pop):,}")
print(pop["population_comparability"].value_counts())

Comuni 2025: 7,896
population_comparability
comparable_on_2025_geography                    7894
not_comparable_due_to_2021_territorial_split       2
Name: count, dtype: int64


In [3]:
numeric = ["population_2019","population_2025","population_change_absolute","population_change_percent",
           "population_0_14_2019","population_15_64_2019","population_65_plus_2019","share_65_plus_2019",
           "population_0_14_2025","population_15_64_2025","population_65_plus_2025","share_65_plus_2025"]
for c in numeric:
    pop[c] = pd.to_numeric(pop[c], errors="coerce")
pop[numeric].describe().T

,count,mean,std,min,25%,50%,75%,max
population_2019,7894.0,7569.088422,42551.755119,32.000000,1016.250000,2471.000000,6322.500000,2.820219e+06
population_2025,7896.0,7464.977710,41542.470999,32.000000,969.000000,2383.000000,6239.250000,2.747290e+06
population_change_absolute,7894.0,-110.252090,1164.500214,-72929.000000,-132.000000,-46.000000,6.000000,8.123000e+03
population_change_percent,7894.0,-2.875617,5.106873,-43.750000,-5.915848,-2.527954,0.314198,4.545455e+01
population_0_14_2019,7894.0,996.123258,5558.871201,0.000000,110.000000,303.000000,836.000000,3.702180e+05
population_15_64_2019,7894.0,4840.232455,27273.831836,17.000000,629.250000,1562.000000,4027.500000,1.814684e+06
population_65_plus_2019,7894.0,1732.732708,9769.668164,7.000000,268.250000,585.000000,1404.500000,6.353170e+05
share_65_plus_2019,7894.0,25.046302,5.405057,8.419476,21.491299,24.498671,27.915943,6.438356e+01
population_0_14_2025,7896.0,889.063956,4913.339598,0.000000,96.000000,266.000000,746.250000,3.249170e+05
population_15_64_2025,7896.0,4729.426039,26666.558902,16.000000,591.750000,1490.500000,3941.000000,1.763502e+06


In [4]:
noncomp = pop.loc[pop["population_comparability"] != "comparable_on_2025_geography", ["istat_code","municipality_name","population_comparability"]]
assert len(noncomp) == 2
noncomp

,istat_code,municipality_name,population_comparability
4010,081025,Misiliscemi,not_comparable_due_to_2021_territorial_split
7210,081021,Trapani,not_comparable_due_to_2021_territorial_split
